In [3]:
import rasterio as rio 


In [4]:
path = 'data/rsimg/s2_20251206.tif'
path_clip = 'data/rsimg/s2_20251206_clip_1.tif'
clip_extent = [266675, 2733747, 270892, 2737296]  # Replace with actual coordinates



In [5]:
with rio.open(path) as src:
      window = src.window(*clip_extent)
      window = window.round_offsets().round_lengths()
      data = src.read(window=window)
      new_transform = src.window_transform(window)

      with rio.open(
            fp=path_clip, 
            mode='w', 
            driver=src.driver, 
            height=window.height,
            width=window.width,
            count=src.count,
            dtype=src.dtypes[0],
            crs=src.crs,
            transform=new_transform) as dst:
         dst.write(data) 





In [17]:
## (2) 基于矢量裁剪 (创建区域矢量)
import geopandas as gpd
import rasterio as rio
from rasterio.mask import mask


In [7]:
path = 'data/rsimg/s2_20251206.tif'
path_ynu_vec = 'data/chenggong.gpkg'
path_rsimg_clip = 'data/rsimg/s2_20251206_clip_2.tif'   




In [11]:
chenggong_gpd = gpd.read_file(path_ynu_vec)
chenggong_polygons = chenggong_gpd.to_crs(epsg=32648).geometry.values
chenggong_polygons


<GeometryArray>
[<MULTIPOLYGON (((282807.685 2733200.548, 282654.195 2733263.118, 282394.374 ...>]
Length: 1, dtype: geometry

In [18]:

with rio.open(path) as src:
    # 执行掩膜裁剪
    new_data, new_transform = mask(
        src,
        chenggong_polygons,
        crop=True,           # 裁剪到最小外接矩形
        all_touched=False,   # 只包含完全在内部的像素
    )

    with rio.open(
            path_rsimg_clip, 
            'w',
            driver=src.driver,
            height=new_data.shape[1],
            width=new_data.shape[2],
            count=new_data.shape[0],
            dtype=src.dtypes[0],
            crs=src.crs,
            transform=new_transform
        ) as dst:
        dst.write(new_data)


In [21]:
import rasterio
from rasterio.merge import merge


In [20]:
# 输入文件列表
path_rsimg_1 = 'data/rsimg/s2_20251206_clip1.tif'
path_rsimg_2 = 'data/rsimg/s2_20251206_clip2.tif'
path_mosaic = 'data/rsimg/s2_20251206_mosaic.tif'




In [22]:
# 读取所有栅格
src_1 = rio.open(path_rsimg_1)
src_2 = rio.open(path_rsimg_2)

# 执行拼接
merged_data, merged_transform = merge([src_1, src_2])


In [23]:
## 写入结果
with rasterio.open(
            fp=path_mosaic, 
            mode="w", 
            driver="GTiff",
            height=merged_data.shape[1],
            width=merged_data.shape[2],
            count=src_1.count,
            dtype=src_1.dtypes[0],
            crs=src_1.crs,
            transform=merged_transform) as dst:
    dst.write(merged_data)

# 关闭所有输入文件
src_1.close()
src_2.close() 


